# Lab 02 — Anomaly Detection & Alerting

In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/self-study/diagrams/lab02_pipeline_position.svg"))
(PROJECT_ROOT / "outputs" / "workshop").mkdir(parents=True, exist_ok=True)

## Lab 總覽

### 學習目標
1. 理解三種偵測哲學：固定閾值、統計基線（Z-score）、change point 偵測
2. 實作固定閾值偵測並理解其局限
3. 實作 Z-score detection + 滯帶（deadband）防抖機制
4. 理解並實作簡易 change point 偵測（雙均線發散法）
5. 將偵測結果輸出為 Grafana 可讀格式

### 前置條件
- 已完成 Lab 01（或可從 `/tmp/lab01_features.csv` 讀取特徵資料）
- `pandas`、`matplotlib`、`numpy` 已安裝

## 設計主線：把偵測結果變成可用告警

本章的目標不是找出最多紅點，而是設計一個工程師願意相信的告警策略。固定閾值、Z-score、deadband、change point 各自適合不同異常形狀。

請用三個實務問題檢查 detection design：

1. **告警是否可解釋**：工程師收到 alert 時，能否知道是哪個 metric、哪個 baseline、哪個 threshold 觸發？
2. **靈敏度是否符合值班負擔**：越敏感不一定越好。誤報會消耗注意力，漏報會放大事件損失。
3. **輸出是否能被下游使用**：event list 要包含 timestamp、device、metric、severity，否則 RCA 和 Grafana annotation 都接不住。

設計原則：偵測器不是模型展示，而是告警系統的一個元件。


## 本章在 Pipeline 的位置

圖表說明：本章（Lab 02）接收 Lab 01 輸出的特徵向量，套用三種偵測策略（固定閾值、Z-score、change point 偵測），輸出事件清單供 Grafana Annotations 使用，並作為 Lab 08 RCA 的上游輸入。

偵測策略沒有通用最優解——每種方法對不同異常形狀有不同靈敏度，閾值選擇是業務決定，不是技術決定。

---
**概念說明 ▸ 講師引導** — 請聆聽講師說明三種偵測哲學，完成後再執行以下 cell。

---

## 概念：三種偵測哲學

（見下方比較圖）

---

### 三種方法的核心假設

每種偵測方法都基於一個關於「什麼是異常」的假設，了解假設才能知道何時方法會失效：

```
Method           Core assumption                          When assumption fails

Fixed threshold  "Normal value is always below X"         System growth / seasonal traffic → false alerts
Z-score          "Normal values follow a normal dist."     Heavy-tailed traffic distribution → misses genuine anomalies
Change point     "System is stationary; only breaks are anomalous"  Slow continuous drift → undetectable
```

---

### 何時用哪種方法？

```
Your question:                                          Recommended method

"A value must never exceed X (SLA upper bound)"     →  Fixed threshold (simplest; start here)
"The value is suddenly much higher than usual"       →  Z-score (most general-purpose)
"The system's behavior pattern has suddenly changed" →  Change point detection (detects regime shift)
"Need to detect slow drift"                         →  CUSUM / change point detection
"There is time periodicity (day/night, weekly)"     →  Seasonal decomposition + Z-score
"Don't know what 'normal' looks like"               →  Machine learning (unsupervised, Lab 04)
```

---

### 真實環境常見的混用策略

單一方法在真實環境通常不夠，實務上多層疊加：

```
Layer 1 (fastest): fixed threshold
  → immediate alert on extreme values (e.g. traffic > 95% of link capacity)

Layer 2 (adaptive): Z-score + deadband
  → detect spikes deviating from baseline; filter transient fluctuations

Layer 3 (trend): change point detection
  → detect long-term regime drift; give early warning of capacity issues

Each layer filters further, minimizing alert count while maximizing quality for the human reviewer.
```

---

**參考資料：**
- Z-score 說明：https://en.wikipedia.org/wiki/Standard_score
- change point detection：https://en.wikipedia.org/wiki/Change_detection
- ruptures 函式庫（進階）：https://centre-borelli.github.io/ruptures-docs/


In [ ]:
from pathlib import Path
from IPython.display import SVG

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "environment.yml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SVG(filename=str(PROJECT_ROOT / "labs/workshop/diagrams/lab02_detection_methods.svg"))

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 0：載入資料與環境準備

In [ ]:
import urllib.request
import urllib.parse
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11

PROM = "http://localhost:9090"

def prom_range(expr, hours=6, step="1m"):
    end = datetime.utcnow()
    start = end - timedelta(hours=hours)
    params = urllib.parse.urlencode({
        "query": expr,
        "start": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "end":   end.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "step":  step,
    })
    with urllib.request.urlopen(f"{PROM}/api/v1/query_range?{params}", timeout=8) as r:
        return json.loads(r.read())

def to_df(result, value_col="value"):
    rows = []
    for series in result["data"]["result"]:
        dev = series["metric"].get("device",
              series["metric"].get("instance", "?"))
        for ts, val in series["values"]:
            rows.append({"timestamp": pd.Timestamp(ts, unit="s"),
                          "device": dev, value_col: float(val)})
    return pd.DataFrame(rows)

print("✅ 環境準備完成")

In [ ]:
def generate_synthetic_with_anomalies(n_minutes=360):
    """生成含有三種典型異常的合成資料。"""
    np.random.seed(2024)
    end = datetime.utcnow()
    timestamps = [pd.Timestamp(end - timedelta(minutes=n_minutes - i)) for i in range(n_minutes)]
    t = np.linspace(0, 2 * np.pi, n_minutes)
    
    # 基準：日間Traffic曲線
    base = 2_000_000 + 800_000 * np.sin(t * 0.5)
    noise = np.random.normal(0, 150_000, n_minutes)
    rx = np.maximum(base + noise, 0)
    
    # 異常 1：瞬間突刺（Traffic爆炸）— 分鐘 90-91
    rx[90:92] *= 8
    
    # 異常 2：持續偏高（系統狀態改變）— 分鐘 180-210
    rx[180:210] *= 3.5
    
    # 異常 3：Traffic驟降（連線中斷）— 分鐘 280-285
    rx[280:285] *= 0.05
    
    return pd.DataFrame({
        'timestamp': timestamps,
        'device': 'eth0',
        'rx_rate': rx,
        'tx_rate': rx * 0.25 + np.random.normal(0, 20_000, n_minutes),
    })

# 嘗試從 Lab 01 儲存的特徵讀取，否則從 Prometheus 拉取，否則用合成資料
_lab01_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_lab01_features.csv"
if _lab01_path.exists():
    df_raw = pd.read_csv(_lab01_path, parse_dates=['timestamp'])
    df_raw = df_raw.sort_values('timestamp').reset_index(drop=True)
    print(f"✅ 從 Lab 01 特徵讀取：{len(df_raw)} 筆")
else:
    filter_expr = 'device!~"lo|docker.*|veth.*|br-.*"'
    try:
        r_rx = prom_range(
            f'rate(node_network_receive_bytes_total{{{filter_expr}}}[1m])',
            hours=6, step='1m'
        )
        r_tx = prom_range(
            f'rate(node_network_transmit_bytes_total{{{filter_expr}}}[1m])',
            hours=6, step='1m'
        )
        df_rx = to_df(r_rx, 'rx_rate')
        df_tx = to_df(r_tx, 'tx_rate')
        df_raw = df_rx.merge(df_tx[['timestamp', 'device', 'tx_rate']],
                             on=['timestamp', 'device'], how='left').fillna(0)
        df_raw = df_raw.sort_values(['device', 'timestamp']).reset_index(drop=True)
        print(f"✅ Prometheus：取得 {len(df_raw)} 筆資料")
    except Exception as e:
        print(f"⚠ Prometheus 無法連線（{e}）— 使用含異常的合成資料")
        df_raw = generate_synthetic_with_anomalies()
        print(f"   合成資料：{len(df_raw)} 筆（含 3 種典型異常）")

# 使用主要裝置
DEVICE = df_raw['device'].value_counts().index[0]
df = df_raw[df_raw['device'] == DEVICE].copy().sort_values('timestamp').reset_index(drop=True)
df['traffic_total'] = df['rx_rate'] + df['tx_rate']
print(f"\n分析裝置：{DEVICE}，筆數：{len(df)}")

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 1：固定閾值偵測

### 概念：固定閾值是偵測的基礎

固定閾值是最簡單、最快速的異常偵測方式：設定一個不能超過的值，超過就告警。

```
Traffic (MB/s)
  ↑
8 │────────────────────────────── fixed threshold = 8 MB/s
  │                     ╱╲
  │                    ╱  ╲   ← alert triggered
  │ ───────────────────    ───────
  └─────────────────────────────→ Time
```

**閾值設定方式（本 Lab 採用）：**
用「基線期間的 95th percentile」作為閾值：
- 取前 20% 資料作為基線期間（「正常狀態」的樣本）
- 計算這段期間的 95th percentile
- 設定為「正常值上限」，超過即告警

這比「直接拍一個數字」更有統計依據，也比靜態設定更能適應不同環境。

**PromQL 等效（可直接部署到 Prometheus）：**
```promql
rate(node_network_receive_bytes_total{device!~"lo|docker.*"}[1m])
  > 8 * 1048576  # 8 MB/s
```


In [ ]:
# threshold設定：以基線期間（前 20% 資料）的 95th percentile 為threshold
baseline_len = max(int(len(df) * 0.2), 10)
baseline_period = df.iloc[:baseline_len]

THRESHOLD_HIGH = baseline_period['traffic_total'].quantile(0.95)
THRESHOLD_LOW  = baseline_period['traffic_total'].quantile(0.05)

print(f"基線期間：前 {baseline_len} 筆（{df['timestamp'].iloc[0]} ~ {df['timestamp'].iloc[baseline_len-1]}）")
print(f"基線mean：    {baseline_period['traffic_total'].mean()/1e6:.2f} MB/s")
print(f"高threshold (P95)：{THRESHOLD_HIGH/1e6:.2f} MB/s")
print(f"低threshold (P05)：{THRESHOLD_LOW/1e6:.2f} MB/s")

df['flag_high'] = df['traffic_total'] > THRESHOLD_HIGH
df['flag_low']  = df['traffic_total'] < THRESHOLD_LOW
df['flag_threshold'] = df['flag_high'] | df['flag_low']

high_count = df['flag_high'].sum()
low_count  = df['flag_low'].sum()
print(f"\n偵測結果：")
print(f"  high traffic告警：{high_count} 筆 ({high_count/len(df)*100:.1f}%)")
print(f"  low traffic告警：{low_count} 筆 ({low_count/len(df)*100:.1f}%)")
print(f"  合計：{df['flag_threshold'].sum()} 筆")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df['timestamp'], df['traffic_total'] / 1e6,
        color='steelblue', linewidth=1, alpha=0.8, label='Traffic')

# threshold線
ax.axhline(THRESHOLD_HIGH / 1e6, color='red', linestyle='--',
           linewidth=2, label=f'high threshold {THRESHOLD_HIGH/1e6:.2f} MB/s')
ax.axhline(THRESHOLD_LOW / 1e6, color='orange', linestyle='--',
           linewidth=2, label=f'low threshold {THRESHOLD_LOW/1e6:.2f} MB/s')

# 標記異常區域
ax.fill_between(df['timestamp'], df['traffic_total'] / 1e6, THRESHOLD_HIGH / 1e6,
                where=df['flag_high'], alpha=0.4, color='red', label='high-traffic anomaly')
ax.fill_between(df['timestamp'], THRESHOLD_LOW / 1e6, df['traffic_total'] / 1e6,
                where=df['flag_low'], alpha=0.4, color='orange', label='low-traffic anomaly')

# 標記基線期間
ax.axvspan(df['timestamp'].iloc[0], df['timestamp'].iloc[baseline_len-1],
           alpha=0.1, color='green', label='baseline period')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_ylabel('Traffic (MB/s)')
ax.set_title('Fixed threshold detection')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 固定閾值的取捨分析

| 面向 | 評估 |
|------|------|
| 優點 | 極簡單、可解釋、PromQL 一行實現 |
| 優點 | 不需要歷史資料量（立即可用）|
| 缺點 | 流量有日夜週期性 → 夜間的「正常高峰」可能誤觸發 |
| 缺點 | 閾值靜態 → 系統擴容後需手動更新 |
| 何時選擇 | 有硬性 SLA 上限、快速 POC、初始監控 |
| 替代方案 | Z-score（動態基線）、Isolation Forest（無監督）|

> **PromQL 等效實作：**
> ```promql
> rate(node_network_receive_bytes_total{device!~"lo|docker.*"}[1m]) > 5000000
> ```

---
**自訂練習** — 先決定你的閾值設計策略，再填入數值。

---

## 🧠 設計思考：你的固定閾值應該設在哪裡？

---

### Q1：你的鏈路容量是多少？「80% 規則」適用嗎？

```
The most intuitive approach to setting a fixed threshold:
  threshold = link capacity × 80%

Underlying assumption: link utilization above 80% indicates potential congestion or anomaly

Real-world case — why the 80% rule is not precise enough:
  A university's 10 Gbps campus backbone carries traffic that ramps from 0.8 Gbps to 7.2 Gbps when classes start at 9:00 AM.
  Normal peak utilization = 7.2 / 10 = 72% (below 80%).
  One day a department held a large-scale online course enrollment; traffic spiked to 9.1 Gbps (91%), causing packet loss.
  → Fixed 80% threshold = 8 Gbps → correctly triggered at 9.1 Gbps ✓
  But the same threshold fired 50 times on the semester-opening results day (traffic = 10 Gbps).
  → Root problem: the threshold ignores "scheduled events" and cannot distinguish planned high traffic from a genuine anomaly
```

---

### Q2：你知道歷史資料中的正常峰值嗎？

```
Percentile method (more precise than the 80% rule):

  Step 1: collect 2–4 weeks of historical data; keep only "normal periods" (exclude known large events)
  Step 2: compute the 95th or 99th percentile
  Step 3: set that value as the threshold

Real-world case — ISP data-driven threshold calibration:
  An ISP analyzed 30 days of backbone link data:
  p50 = 4.2 GB/s  (daily baseline)
  p90 = 7.8 GB/s  (workday peak)
  p95 = 8.9 GB/s  (peak hour)
  p99 = 11.2 GB/s (maximum instantaneous demand)
  link capacity = 14 GB/s

  Engineers set threshold = p99 × 1.1 = 12.3 GB/s
  Rationale: anything above p99 is almost certainly anomalous traffic, not a normal peak.

  Result: only 3 alerts fired in the following month; all three were genuine traffic anomalies or device failures.
```

---

### Q3：你的告警預算是多少？

```
Inverse thinking: how many false positives can your on-call team tolerate?

  Assumption: on a normal day, the probability that traffic exceeds the threshold is p
              alert trigger count = p × daily data points (determined by scrape_interval)

  Scenario calculation (1 sample/min, 1440 samples/day):
  p = 1%    → ~14 triggers per day
  p = 0.5%  → ~7 triggers per day
  p = 0.1%  → ~1.4 triggers per day (acceptable alert frequency for engineers)

  Conclusion: setting the threshold at the p99.9 percentile achieves roughly 1 false alert per day
```

---

### Q4：閾值設了之後，多久需要重新校準？

```
Real-world case — forgot to update threshold after capacity upgrade:
  A company upgraded its external link from 1 Gbps to 10 Gbps at end of 2024.
  Old threshold = 800 Mbps (80% of 1 Gbps).
  The threshold was not adjusted after the upgrade.
  Result: alerts triggered almost every day on the new link (normal traffic 1.5–3 Gbps far exceeded the old threshold)
  → Alert fatigue: on-call engineers started ignoring alerts; a real incident 3 months later went unnoticed.

  Recommendation: establish a "threshold audit cycle"
  - Audit threshold within 7 days of any capacity change
  - Recompute percentiles quarterly
  - If on-call receives more than X alerts in one week, force a threshold audit
```

---

## ✏ 自訂固定閾值偵測

想清楚上面四個問題後，在下方 cell 設定 `FLAG_COL` 和 `THRESHOLD_MBS`。


In [ ]:
# ── 1. 為你的偵測旗標命名 ──────────────────────────────────────────────
FLAG_COL = 'flag_threshold'   # ← 改成你想要的欄位名稱

# ── 2. 設定閾值 ──────────────────────────────────────────────────────────────
THRESHOLD_MBS = 6.0    # ← 你認為合理的上限（MB/s）

# 或用百分位數自動設定（取消下面兩行）：
# PERCENTILE    = 90
# THRESHOLD_MBS = df['traffic_total'].quantile(PERCENTILE / 100) / 1e6

# ── 計算 ────────────────────────────────────────────────────────────────────
THRESHOLD = THRESHOLD_MBS * 1e6
df[FLAG_COL] = df['traffic_total'] > THRESHOLD

n    = df[FLAG_COL].sum()
rate = n / len(df) * 100
print(f"旗標欄位：{FLAG_COL}  |  閾值：{THRESHOLD_MBS:.1f} MB/s")
print(f"觸發：{n} 筆（佔 {rate:.1f}%）")


In [ ]:
# 視覺化你的固定閾值偵測結果
fig, ax = plt.subplots(figsize=(12, 4))

ax.plot(df['timestamp'], df['traffic_total'] / 1e6,
        linewidth=0.8, color='steelblue', label='traffic_total')
ax.axhline(THRESHOLD_MBS, color='red', linestyle='--', linewidth=1.2,
           label=f'{FLAG_COL} threshold = {THRESHOLD_MBS} MB/s')

hit = df[df[FLAG_COL]]
ax.scatter(hit['timestamp'], hit['traffic_total'] / 1e6,
           color='red', s=20, zorder=5, label=f'triggered ({len(hit)} points)')

ax.set_ylabel('MB/s');  ax.legend(fontsize=9)
ax.set_title(f'{FLAG_COL}  threshold = {THRESHOLD_MBS} MB/s')
plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
print(f"\n📊 固定閾值偵測量化（{FLAG_COL}，threshold = {THRESHOLD_MBS} MB/s）")
print("─" * 55)

total = len(df)
hit_count = int(df[FLAG_COL].sum())
miss_count = total - hit_count
print(f"  總資料點   ：{total} 筆")
print(f"  觸發（True）：{hit_count} 筆 ({100*hit_count/total:.2f}%)")
print(f"  未觸發     ：{miss_count} 筆 ({100*miss_count/total:.2f}%)")

# Alert event clustering (consecutive=1 event)
import numpy as np
flags = df[FLAG_COL].values.astype(int)
changes = np.diff(np.concatenate([[0], flags, [0]]))
event_starts = np.where(changes == 1)[0]
event_ends   = np.where(changes == -1)[0]
n_events = len(event_starts)
durations = event_ends - event_starts  # in minutes (1 point = 1 min)

print(f"\n  告警事件（連續觸發算 1 次）：")
print(f"    事件數量  ：{n_events} 次")
if n_events > 0:
    print(f"    最短持續  ：{durations.min()} 分鐘")
    print(f"    最長持續  ：{durations.max()} 分鐘")
    print(f"    平均持續  ：{durations.mean():.1f} 分鐘")
    print(f"    告警率    ：{n_events / (total/60):.1f} 次 / 小時")

print(f"\n  🔎 自動診斷：")
alert_rate_per_hour = n_events / (total / 60)
traffic_pct = THRESHOLD_MBS / (df['traffic_total'].max() / 1e6) * 100
print(f"    閾值佔最大流量的 {traffic_pct:.1f}%")

if alert_rate_per_hour > 5:
    print(f"    ⚠️  告警頻率 {alert_rate_per_hour:.1f} 次/小時，偏高。")
    print(f"       建議：提高閾值 or 加入 Lab 02 Step 3 的 Deadband 防抖")
elif alert_rate_per_hour < 0.1 and total > 120:
    print(f"    ⚠️  6 小時內只觸發 {n_events} 次，閾值可能太高（太難觸發）")
elif n_events == 0:
    print(f"    ❌  0 次觸發：閾值高於資料集的所有觀測值，偵測不到任何異常")
else:
    print(f"    ✅ 告警頻率 {alert_rate_per_hour:.1f} 次/小時，閾值設定合理")

# Compare with auto-derived baseline threshold
baseline_p95 = float(np.percentile(df['traffic_total'].dropna(), 95) / 1e6)
print(f"\n  參考：資料集 p95 = {baseline_p95:.2f} MB/s（統計閾值參考）")
if THRESHOLD_MBS < baseline_p95 * 0.8:
    print(f"    ⚠️  你的閾值低於 p95×0.8，正常高峰可能會被誤報")
elif THRESHOLD_MBS > baseline_p95 * 1.5:
    print(f"    ⚠️  你的閾值高於 p95×1.5，真實異常可能被遺漏")
else:
    print(f"    ✅ 閾值在 p95 的合理倍率範圍內")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2：Z-score 異常偵測

Z-score 衡量「當前值偏離移動基線幾個標準差」：

```
Formula:
  Z = (x − μ) / σ

  x = current value
  μ = rolling mean (average over the past WINDOW minutes)
  σ = rolling std  (volatility over the past WINDOW minutes)

Example:
  μ = 5.0 MB/s (mean traffic over the last 20 minutes)
  σ = 0.5 MB/s (volatility over the last 20 minutes)
  x = 7.0 MB/s (current traffic)
  Z = (7.0 − 5.0) / 0.5 = 4.0  ← 4 standard deviations above baseline
```

**常態分佈下的期望比例：**

```
        68%
     ◄───────►
          95%
       ◄────────────►
              99.7%
    ◄───────────────────►

  -3σ  -2σ  -1σ   0   +1σ  +2σ  +3σ

  → Z > 3: under a perfect normal distribution, this occurs less than 0.3% of the time
  → Network traffic is typically heavy-tailed (right-skewed); in practice Z > 3 occurs more often
  → This is why deadband (N consecutive points) is more reliable than single-point triggering
```

---

### Z-score 的兩個已知失效場景

**1. 窗口被異常污染（baseline drift）：**
```
  Anomaly lasts longer than WINDOW → rolling_mean catches up → Z-score falls back

  Solution: use a longer WINDOW, or combine with EWMA (exponentially weighted moving average)
```

**2. 流量分佈是重尾的：**
```
  Traffic occasionally has extreme spikes (e.g. large file transfers), pulling std upward
  → Small but sustained genuine anomalies may have Z-score below 3

  Solution: apply log transform to traffic_total before computing Z-score (makes distribution closer to normal)
```

這些限制是 Lab 02 要介紹三種方法的原因——沒有萬能的偵測方法。


In [ ]:
WINDOW = 20   # 滾動window（分鐘）
Z_THRESH = 3.0  # 異常閾值

df['roll_mean'] = df['traffic_total'].rolling(WINDOW, min_periods=3).mean()
df['roll_std']  = df['traffic_total'].rolling(WINDOW, min_periods=3).std().fillna(1)
df['z_score']   = (df['traffic_total'] - df['roll_mean']) / (df['roll_std'] + 1e-9)

df['flag_zscore_raw'] = df['z_score'].abs() > Z_THRESH

print(f"Z-score detection結果（window={WINDOW}min, Z>{Z_THRESH}）：")
print(f"  原始異常標記：{df['flag_zscore_raw'].sum()} 筆")
print(f"  最大 Z-score：{df['z_score'].max():.2f}")
print(f"  最小 Z-score：{df['z_score'].min():.2f}")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(df['timestamp'], df['traffic_total'] / 1e6, color='steelblue', linewidth=1, alpha=0.7, label='Traffic')
ax.plot(df['timestamp'], df['roll_mean'] / 1e6, color='navy', linewidth=2, label=f'{WINDOW}min mean')
ax.fill_between(
    df['timestamp'],
    (df['roll_mean'] - Z_THRESH * df['roll_std']) / 1e6,
    (df['roll_mean'] + Z_THRESH * df['roll_std']) / 1e6,
    alpha=0.15, color='navy', label=f'±{Z_THRESH}σ band'
)
anomalies_z = df[df['flag_zscore_raw']]
ax.scatter(anomalies_z['timestamp'], anomalies_z['traffic_total'] / 1e6,
           color='red', s=60, zorder=5, label=f'Z-score anomaly ({len(anomalies_z)})')
ax.set_ylabel('MB/s')
ax.set_title(f'Z-score detection（window={WINDOW}min）')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(df['timestamp'], df['z_score'], color='darkorange', linewidth=1)
ax2.axhline(Z_THRESH,  color='red', linestyle='--', alpha=0.7, label=f'Z=±{Z_THRESH}')
ax2.axhline(-Z_THRESH, color='red', linestyle='--', alpha=0.7)
ax2.axhline(0, color='gray', alpha=0.3)
ax2.scatter(anomalies_z['timestamp'], anomalies_z['z_score'],
            color='red', s=50, zorder=5)
ax2.set_ylabel('Z-score')
ax2.set_title('Z-score time series')
ax2.legend()
ax2.grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3：Deadband（滯帶）防抖 — 避免告警抖動（Flapping）

### 告警疲勞（Alert Fatigue）是真實問題

```
Phenomenon: Z-score repeatedly crosses the threshold → causes alert flapping

  Z-score
  ↑
3 │──────────────────────── Z=3 threshold
  │      ↑↑↑↑↑↑↑
  │   ↑↑      ↑↑↑↑       ← crosses threshold repeatedly
  └─────────────────────→ Time
  Alerts:  on off on off on off   ← one event generates 6 notifications!

Result:
  On-call engineer woken by PagerDuty 6 times
  Starts ignoring the alert from the 3rd notification onward
  When a genuine anomaly arrives, it is also ignored (alert fatigue)
```

**研究數據：** 約 30–70% 的告警是誤警報或重複警報。告警疲勞是 SRE 領域的重要問題。

---

### Deadband 解法：要求連續 N 筆超過閾值

```
N=3 (alert only after 3 consecutive minutes above threshold):

  Z-score
  ↑
3 │──────────────────────── Z=3 threshold
  │      ↑↑↑↑↑↑↑           ← 3 consecutive exceedances → triggered
  │   ↑↑      ↑↑↑↑
  └─────────────────────→ Time
  Alerts:        on         ← one clean alert

Lowering N (N=1): most sensitive, but most alerts
Raising N (N=5):  most stable, but response latency = N × scrape_interval
```

---

### Deadband 的本質

Deadband 是一種「多數決」機制：過去 N 個時間點都同意有問題，才告警。
這對抗的是感測器噪音——單點異常可能是網卡暫時的計數誤差，不是真正的流量異常。

實務中 N 的選擇依賴你對「真實異常最短持續時間」的理解：
- DDoS 攻擊通常持續 > 1 分鐘 → N=3 合理
- 瞬時封包遺失（< 1 分鐘）→ N=1 才能捕捉

這又是一個人類需要做的業務決策，演算法無法自動決定。


In [ ]:
DEADBAND_N = 3  # 連續 N 筆超過threshold才觸發（試試 1, 3, 5）

# 計算滑動window：過去 N 筆中有幾筆超過threshold
df['consecutive_high'] = (
    df['flag_zscore_raw'].astype(int)
    .rolling(window=DEADBAND_N, min_periods=DEADBAND_N)
    .sum()
)
df['flag_zscore_deadband'] = df['consecutive_high'] >= DEADBAND_N

raw_count = df['flag_zscore_raw'].sum()
db_count  = df['flag_zscore_deadband'].sum()
print(f"Deadband N={DEADBAND_N} 效果：")
print(f"  無 deadband：{raw_count} 筆告警")
print(f"  有 deadband：{db_count} 筆告警")
print(f"  減少：{raw_count - db_count} 筆（{(1 - db_count/max(raw_count,1))*100:.0f}% 減少）")

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

for ax, flag_col, title in [
    (axes[0], 'flag_zscore_raw',      f'無 deadband（{raw_count} 告警）'),
    (axes[1], 'flag_zscore_deadband', f'有 deadband N={DEADBAND_N}（{db_count} 告警）'),
]:
    ax.plot(df['timestamp'], df['z_score'], color='darkorange', linewidth=1)
    ax.axhline(Z_THRESH,  color='red', linestyle='--', alpha=0.5)
    ax.axhline(-Z_THRESH, color='red', linestyle='--', alpha=0.5)
    flagged = df[df[flag_col]]
    if len(flagged) > 0:
        ax.scatter(flagged['timestamp'], flagged['z_score'],
                   color='red', s=40, zorder=5)
    ax.set_ylabel('Z-score')
    ax.set_title(title)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
**自訂練習** — 先設計參數優先順序，再動手調整數值。

---

## 🧠 設計思考：Z-score + Deadband 的三步設計順序

最常見的錯誤：先設 Z_THRESH，再調 WINDOW，最後加 DEADBAND。
這個順序會讓你反覆修改。正確順序是反過來的。

---

### 第一步：決定 WINDOW（基線記憶長度）

**核心問題：你的最長正常波動持續多久？**

```
Real-world case A — Telecom backbone link (WINDOW = 60 minutes):
  Normal traffic on a workday has a ~40-minute ramp-up starting at 9 AM (employees arriving).
  With WINDOW = 20 minutes, every morning peak triggers an alert.
  After extending WINDOW to 60 minutes, the baseline tracks the ramp-up and Z-score no longer triggers on the commute peak.
  Key point: WINDOW must be ≥ the longest normal fluctuation duration

Real-world case B — Factory equipment monitoring (WINDOW = 5 minutes):
  Factory equipment has a 3–5 minute machining cycle; traffic appears as periodic spikes.
  WINDOW = 60 minutes causes the entire baseline to include the spikes → Z-score stays below 1 → genuine anomalies undetectable.
  Shrinking to WINDOW = 5 minutes makes anomalous spikes within each cycle clearly visible.
  Key point: WINDOW must not be much longer than the time scale of the anomaly you want to detect
```

---

### 第二步：決定 Z_THRESH（告警靈敏度）

**核心問題：FP（誤報）和 FN（漏報）哪個代價更高？**

```
Comparison of three scenarios:

  Financial institution API gateway (FN cost is very high):
    One undetected DDoS = transaction outage = financial loss
    → Set Z_THRESH = 2.0 (accept more false positives rather than miss attacks)
    → May see 5–10 false alerts per day, but genuine attacks will not be missed

  Large public cloud batch jobs (FP cost is very high):
    False alert wakes engineers in the middle of the night to investigate "a perfectly normal large batch job"
    → Set Z_THRESH = 3.5 or even 4.0 (accept missing some minor anomalies rather than cause alert fatigue)

  General enterprise operations:
    → Z_THRESH = 3.0 is a statistically sound starting point (~0.27% false-positive rate under a normal distribution)
    → Start here; observe for one week before tuning

Auto-calibration method (common in practice):
  Run with WINDOW = 30 minutes and Z_THRESH = 0; collect the full Z-score distribution
  Find the upper edge of the "noise floor" (e.g. p95 is Z = 2.1)
  Set Z_THRESH = noise floor upper edge × 1.5 = 3.15 → stay above noise, capture genuine anomalies
```

---

### 第三步：決定 DEADBAND（連續確認次數）

**核心問題：最短的「真實事件」持續多久？**

```
Real-world case — how deadband saved the on-call engineer's sleep:
  An e-commerce platform used Z_THRESH = 2.5; alerts fired ~30 times per day.
  25 of those were 1–2-minute transient spikes (log cleanup, cache flush).
  On-call received 200 alerts per week, mostly false positives, and started ignoring them.

  After adding DEADBAND = 3 (alert only after 3 consecutive minutes above Z_THRESH):
  → Daily alerts dropped from 30 to 5
  → 4 of the 5 were genuine traffic anomaly events
  → Engineers started trusting the alert system; response time improved

  Formula:
  DEADBAND = shortest genuine event duration ÷ scrape_interval (minutes)
  Example: shortest genuine event lasts 5 minutes, scrape every 1 minute → DEADBAND = 5
```

---

## ✏ 自訂 Z-score + Deadband 偵測

三步設計完成後，在下方 cell 填入參數。


In [ ]:
# ── 1. 為你的欄位命名 ─────────────────────────────────────────────────
ZSCORE_COL = 'zscore_custom'    # ← Z-score 欄位名稱
FLAG_RAW   = 'flag_z_raw'       # ← 原始旗標（單點超閾值）
FLAG_DB    = 'flag_z_db'        # ← Deadband 後旗標（連續 N 點才觸發）

# ── 2. 設定參數 ──────────────────────────────────────────────────────────────
WINDOW   = 20             # ← 滾動窗口（分鐘）：試試 10, 30, 60
Z_THRESH = 3.0            # ← Z-score 閾值：試試 2.0, 2.5, 4.0
DEADBAND = 3              # ← 連續確認筆數：試試 1, 5
DET_COL  = 'traffic_total'  # ← 偵測目標：試試 'error_rate', 'tx_ratio'

# ── 計算 ────────────────────────────────────────────────────────────────────
_mean = df[DET_COL].rolling(WINDOW).mean()
_std  = df[DET_COL].rolling(WINDOW).std()
df[ZSCORE_COL] = (df[DET_COL] - _mean) / (_std + 1e-9)
df[FLAG_RAW]   = df[ZSCORE_COL] > Z_THRESH
df[FLAG_DB]    = df[FLAG_RAW].astype(int).rolling(DEADBAND).sum() >= DEADBAND

n_raw = df[FLAG_RAW].sum()
n_db  = df[FLAG_DB].sum()
print(f"旗標：{FLAG_DB}  |  WINDOW={WINDOW} Z>{Z_THRESH} deadband={DEADBAND}")
print(f"原始觸發：{n_raw} 筆  →  Deadband 後：{n_db} 筆（過濾掉 {n_raw - n_db} 筆）")


In [ ]:
# 視覺化你的 Z-score 偵測結果
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

valid = df.dropna(subset=[ZSCORE_COL])
_mean_v = valid[DET_COL].rolling(WINDOW).mean()
_std_v  = valid[DET_COL].rolling(WINDOW).std()

axes[0].plot(valid['timestamp'], valid[DET_COL]/1e6,
             linewidth=0.8, alpha=0.6, color='steelblue', label=DET_COL)
axes[0].plot(valid['timestamp'], _mean_v/1e6,
             linewidth=1.5, color='darkorange', label=f'baseline (W={WINDOW}min)')
axes[0].fill_between(valid['timestamp'],
    (_mean_v - Z_THRESH * _std_v)/1e6,
    (_mean_v + Z_THRESH * _std_v)/1e6,
    alpha=0.15, color='darkorange', label=f'±{Z_THRESH}σ')
hit = valid[valid[FLAG_DB]]
axes[0].scatter(hit['timestamp'], hit[DET_COL]/1e6,
                color='red', s=25, zorder=5, label=f'{FLAG_DB} ({len(hit)} points)')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=8)
axes[0].set_title(f'{FLAG_DB}（W={WINDOW}, Z>{Z_THRESH}, N={DEADBAND}）')

axes[1].plot(valid['timestamp'], valid[ZSCORE_COL],
             linewidth=0.8, color='purple', alpha=0.8, label=ZSCORE_COL)
axes[1].axhline( Z_THRESH, color='red', linestyle='--', linewidth=0.8, label=f'Z=±{Z_THRESH}')
axes[1].axhline(-Z_THRESH, color='red', linestyle='--', linewidth=0.8)
axes[1].set_ylabel('Z-score');  axes[1].legend(fontsize=8)

plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
raw_trigger  = int(valid[FLAG_RAW].sum())
db_trigger   = int(valid[FLAG_DB].sum())
total_v      = len(valid)

print(f"\n📊 Z-score + Deadband 量化（W={WINDOW}, Z={Z_THRESH}, N={DEADBAND}）")
print("─" * 55)
print(f"  有效資料點              ：{total_v} 筆")
print(f"  Z 原始觸發（FLAG_RAW）  ：{raw_trigger} 筆 ({100*raw_trigger/max(total_v,1):.2f}%)")
print(f"  Deadband 後（FLAG_DB）  ：{db_trigger} 筆 ({100*db_trigger/max(total_v,1):.2f}%)")
suppressed = raw_trigger - db_trigger
print(f"  Deadband 過濾           ：{suppressed} 筆 ({100*suppressed/max(raw_trigger,1):.1f}%)")

def _count_events(series):
    import numpy as np
    arr = series.values.astype(int)
    ch  = np.diff(np.concatenate([[0], arr, [0]]))
    starts = np.where(ch ==  1)[0]
    ends   = np.where(ch == -1)[0]
    return len(starts), (ends - starts)

n_raw_ev, raw_dur = _count_events(valid[FLAG_RAW])
n_db_ev,  db_dur  = _count_events(valid[FLAG_DB])
print(f"\n  告警事件（連續觸發 = 1 次）：")
print(f"    Deadband 前：{n_raw_ev} 次")
print(f"    Deadband 後：{n_db_ev} 次（消除 {n_raw_ev-n_db_ev} 次短暫抖動）")
if n_db_ev > 0:
    print(f"    DB 事件平均持續：{db_dur.mean():.1f} min（最短 {db_dur.min()} / 最長 {db_dur.max()} min）")

print(f"\n  🔎 自動診斷：")
db_pct = 100 * suppressed / max(raw_trigger, 1)
if n_raw_ev == 0:
    print(f"    ⚠️  Z_THRESH={Z_THRESH} 沒有任何觸發，試試降低 Z_THRESH 或縮小 WINDOW")
elif db_pct > 90:
    print(f"    ⚠️  Deadband 過濾了 {db_pct:.0f}%（>90%），DEADBAND 可能過大，會漏掉短時間事件")
elif db_pct < 10 and n_raw_ev > 5:
    print(f"    ⚠️  Deadband 只過濾了 {db_pct:.0f}%，大多數觸發已是持續性告警，可適度縮小 DEADBAND")
else:
    print(f"    ✅ Deadband 有效過濾 {db_pct:.0f}% 的短暫觸發，保留了持續性事件")


---
**概念說明 ▸ 講師引導** — 請聆聽講師說明change point detection concept。

---

## 概念：change point detection（change point detection）

Z-score 偵測「突刺（spike）」；change point detection「狀態切換（regime shift）」。

```
Z-score detects:                       Change point detection detects:

  Traffic                                Traffic
  ↑    ↑ spike                          ↑
  │    │                                │              ┌──── new baseline (higher)
  │ ~~~│~~~~~~~~~~~~~~~                 │ ─────────────┤
  │    └──                             │              ↑ change point
  └────────────────→ Time              └──────────────────────→ Time

  "One point is anomalously high"        "The system's overall level has shifted"
  (e.g. the instant a DDoS attack starts) (e.g. permanent link quality drop, new service launch)
```

---

### 為什麼 Z-score 偵測不到狀態切換？

```
After a regime shift:

  old baseline = 5 MB/s
  new baseline = 8 MB/s (traffic stays at this level after the shift)
  rolling_mean catches up to the new baseline → Z-score falls back → no longer triggers an alert

  But this "new level" may represent:
    - Network device configuration changed
    - New high-traffic application launched
    - Unauthorized network usage
```

---

### 本 Lab 實作：雙移動均線發散法

```
short MA (6 min) tracks recent changes
long MA (30 min) represents the stable baseline

When short MA diverges significantly from long MA → system state may have shifted

  Traffic
  ↑
  │──────────────┐     ← short MA rising (recent level changed)
  │              │
  │══════════════╪════  ← long MA still in place (historical baseline unmoved)
  │              ↑
  └──────────────────→ Time
                 divergence > 30% → flagged as change point
```

---

### 進階選項（課後自學）

工業級 change point 偵測算法：

| 算法 | 特性 | Python 套件 |
|------|------|-------------|
| CUSUM | 累積和，偵測均值漂移 | `ruptures` |
| PELT | 自動選取 change point 數量 | `ruptures` |
| BOCPD | 貝葉斯線上算法，即時偵測 | `bayesian_changepoint_detection` |

本 Lab 的雙均線法是直觀的教學用近似，實際系統建議評估 `ruptures` 的 PELT。

**參考資料：**
- ruptures 函式庫：https://centre-borelli.github.io/ruptures-docs/


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4：簡易 change point 偵測（雙移動均線發散法）

In [ ]:
# 雙移動均線：短期（6min）vs 長期（30min）
SHORT_WIN  = 6   # 短期window（分鐘）
LONG_WIN   = 30  # 長期window（分鐘）
DIVERGENCE_THRESH = 0.30  # 發散超過 30% → change point

df['ma_short'] = df['rx_rate'].rolling(SHORT_WIN, min_periods=SHORT_WIN // 2).mean()
df['ma_long']  = df['rx_rate'].rolling(LONG_WIN, min_periods=LONG_WIN // 2).mean()

# 發散程度 = |短期 - 長期| / 長期
df['ma_divergence'] = (
    (df['ma_short'] - df['ma_long']).abs() /
    (df['ma_long'].abs() + 1e-9)
)

df['change_point'] = df['ma_divergence'] > DIVERGENCE_THRESH

cp_count = df['change_point'].sum()
print(f"dual moving-average change point detection（短={SHORT_WIN}min, 長={LONG_WIN}min, 發散>{DIVERGENCE_THRESH*100:.0f}%）：")
print(f"  detected change point rows：{cp_count} 筆")

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax = axes[0]
ax.plot(df['timestamp'], df['rx_rate'] / 1e6, color='lightsteelblue', linewidth=1, alpha=0.8, label='RX rate')
ax.plot(df['timestamp'], df['ma_short'] / 1e6, color='steelblue', linewidth=2, label=f'short moving average ({SHORT_WIN}min)')
ax.plot(df['timestamp'], df['ma_long'] / 1e6, color='navy', linewidth=2.5, label=f'long moving average ({LONG_WIN}min)')
cp_mask = df['change_point'] & (~df['change_point'].shift(1).fillna(False))
for ts in df.loc[cp_mask, 'timestamp']:
    ax.axvline(ts, color='red', alpha=0.5, linewidth=1.5)
ax.set_ylabel('MB/s')
ax.set_title('Dual moving-average divergence: red lines = detected change points')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

ax2 = axes[1]
ax2.plot(df['timestamp'], df['ma_divergence'] * 100, color='purple', linewidth=1.5, label='moving-average divergence')
ax2.axhline(DIVERGENCE_THRESH * 100, color='red', linestyle='--', label=f'threshold {DIVERGENCE_THRESH*100:.0f}%')
ax2.fill_between(df['timestamp'], df['ma_divergence'] * 100, DIVERGENCE_THRESH * 100,
                  where=df['change_point'], alpha=0.3, color='red', label='change-point period')
ax2.set_ylabel('Divergence (%)')
ax2.set_title('Moving-average divergence')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()
print(f"\nchange point detection說明：紅線代表系統行為可能已轉換，非單點異常")

### 三種方法比較

| 方法 | 優點 | 缺點 | 適用場景 |
|------|------|------|----------|
| 固定閾值 | 最快實作、PromQL 一行 | 不適應流量波動 | 有 SLA 上限的場景 |
| Z-score + deadband | 自適應基線、抗噪音 | mean/std 需收斂期 | 日常異常偵測 |
| dual moving-average change point detection | 偵測狀態切換 | 延遲較高（要等長期窗口）| 網路品質退化監控 |

---
**自訂練習** — 先理解雙移動均線發散的物理意義，再設計參數。

---

## 🧠 設計思考：change point detection的三個參數如何協同？

change point detection的核心思想：短期均線代表「目前狀態」，長期均線代表「歷史基線」。
當兩者出現持續性發散，說明系統水位發生了根本性的改變。

---

### SHORT_WIN：反應速度 vs 噪音敏感度

```
Smaller SHORT_WIN → short-term MA more sensitive → responds faster to new traffic level
                    → but also more sensitive to instantaneous spikes (false triggers)
Larger SHORT_WIN  → short-term MA smoother → takes longer to confirm a level change
                    → less likely to be falsely triggered by a single spike

Real-world case — permanent traffic increase from new application launch:
  A company deployed a new video streaming feature at 10:00 on Monday.
  Traffic permanently rose from 200 MB/s to 450 MB/s (~2.25×).

  SHORT_WIN = 3 min:  change point detected by 10:05 (within 5 min) ✓ fast
  SHORT_WIN = 20 min: detected at 10:25 (25 min later) ✓ but 25-minute delay
  SHORT_WIN = 60 min: detected at 11:30 (90 min later) ✗ too slow; engineer already found it manually

  ✅ Recommendation: SHORT_WIN = 5–10 minutes is a reasonable starting point for most scenarios
```

---

### LONG_WIN：歷史基線的穩定性要求

```
LONG_WIN must cover at least one complete business cycle to build a reliable baseline:

  Scenario             Business cycle          LONG_WIN recommendation
  ────────────────────────────────────────────────────────────────────────
  Office IT            workday (8 h)           LONG_WIN ≥ 120 min
  E-commerce platform  one day (24 h)          LONG_WIN ≥ 480 min (requires more data)
  Factory equipment    production cycle (4 h)  LONG_WIN ≥ 60 min
  Backbone link (ISP)  one week                LONG_WIN ≥ 1440 min (requires multi-day history)

Real-world case — LONG_WIN too short causes false triggers:
  A media company's live-streaming platform had a prime-time traffic peak every night from 8–10 PM.
  With LONG_WIN = 30 minutes, a change point alert fired at 20:00 every day.
  Reason: the 30-minute baseline straddles the transition from "off-peak" to "peak" level, making it look like a permanent change.
  Extending LONG_WIN to 240 minutes caused the baseline to already include the peak segment → false triggers stopped.
```

---

### DIVERGE_PCT：發散閾值的三步校準法

```
Step 1: set DIVERGE_PCT to 0; observe the natural noise in DIVERGE_COL under normal conditions
Step 2: on normal operating data, compute the p90 or p95 DIVERGE value (this is the noise baseline)
Step 3: set DIVERGE_PCT = noise baseline × 1.5

Real-world case — actual calibration results:
  Core router in an IDC (normal traffic shows ~30% day/night difference):
  Step 1: run with DIVERGE_PCT = 0; collect one day's DIVERGE_COL distribution
  Step 2: p90 = 15%, p95 = 22% (highest during day/night transition)
  Step 3: set DIVERGE_PCT = 22% × 1.5 = 33%

  Result: the weekend-to-Monday workday traffic shift (~35% increase) was correctly detected as a change point.
  The normal daily day/night transition (22%) was excluded from alerts.

Intuition for the SHORT/LONG ratio:
  Smaller SHORT_WIN / LONG_WIN → short-term MA takes longer to move → system is more "inert"
  Ideal ratio: SHORT_WIN ≤ LONG_WIN × 0.2
  Example: LONG_WIN = 30 → SHORT_WIN ≤ 6
```

---

## ✏ 自訂 change point detection

三個參數理清後，在下方 cell 填入你的設計。


In [ ]:
# ── 1. 為你的欄位命名 ─────────────────────────────────────────────────
SHORT_MA_COL = 'short_ma'        # ← 短期均線欄位名稱
LONG_MA_COL  = 'long_ma'         # ← 長期均線欄位名稱
DIVERGE_COL  = 'ma_diverge_pct'  # ← 發散百分比欄位名稱
FLAG_CP      = 'flag_cp'         # ← change point detection flag name

# ── 2. 設定參數 ──────────────────────────────────────────────────────────────
SHORT_WIN   = 6    # ← 短期窗口（分鐘）：試試 3, 10
LONG_WIN    = 30   # ← 長期窗口（分鐘）：試試 15, 60
DIVERGE_PCT = 30   # ← 發散閾值（%）：試試 15, 50
CP_COL      = 'traffic_total'   # ← 偵測目標

# ── 計算 ────────────────────────────────────────────────────────────────────
df[SHORT_MA_COL] = df[CP_COL].rolling(SHORT_WIN).mean()
df[LONG_MA_COL]  = df[CP_COL].rolling(LONG_WIN).mean()
df[DIVERGE_COL]  = ((df[SHORT_MA_COL] - df[LONG_MA_COL]).abs()
                    / (df[LONG_MA_COL] + 1e-9) * 100)
df[FLAG_CP]      = df[DIVERGE_COL] > DIVERGE_PCT

print(f"旗標：{FLAG_CP}  |  SHORT={SHORT_WIN}min  LONG={LONG_WIN}min  閾值={DIVERGE_PCT}%")
print(f"偵測到 change point detection: {df[FLAG_CP].sum()} 筆")


In [ ]:
# 視覺化你的 change point detection結果
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

valid_cp = df.dropna(subset=[SHORT_MA_COL, LONG_MA_COL])

axes[0].plot(valid_cp['timestamp'], valid_cp[CP_COL]/1e6,
             linewidth=0.6, alpha=0.5, color='steelblue', label=CP_COL)
axes[0].plot(valid_cp['timestamp'], valid_cp[SHORT_MA_COL]/1e6,
             linewidth=1.5, color='coral', label=f'{SHORT_MA_COL} ({SHORT_WIN}min)')
axes[0].plot(valid_cp['timestamp'], valid_cp[LONG_MA_COL]/1e6,
             linewidth=1.5, color='seagreen', label=f'{LONG_MA_COL} ({LONG_WIN}min)')
hit_cp = valid_cp[valid_cp[FLAG_CP]]
axes[0].scatter(hit_cp['timestamp'], hit_cp[CP_COL]/1e6,
                color='red', s=20, zorder=5, marker='^',
                label=f'{FLAG_CP} ({len(hit_cp)} points)')
axes[0].set_ylabel('MB/s');  axes[0].legend(fontsize=8)
axes[0].set_title(f'{FLAG_CP}(SHORT={SHORT_WIN}, LONG={LONG_WIN}, threshold={DIVERGE_PCT}%）')

axes[1].plot(valid_cp['timestamp'], valid_cp[DIVERGE_COL],
             linewidth=0.8, color='purple', alpha=0.8, label=DIVERGE_COL)
axes[1].axhline(DIVERGE_PCT, color='red', linestyle='--', linewidth=0.8,
                label=f'threshold = {DIVERGE_PCT}%')
axes[1].set_ylabel('Divergence (%)');  axes[1].legend(fontsize=8)
axes[1].set_title(f'{DIVERGE_COL}(short/long moving-average divergence percent)')

plt.tight_layout();  plt.show()

# ── 結果量化 ───────────────────────────────────────────────────────────────────
import numpy as np

total_cp   = len(valid_cp)
cp_trigger = int(valid_cp[FLAG_CP].sum())
div_vals   = valid_cp[DIVERGE_COL]

print(f"\n📊 change point detection量化(SHORT={SHORT_WIN}, LONG={LONG_WIN}, Pct={DIVERGE_PCT}%）")
print("─" * 60)
print(f"  有效資料點          ：{total_cp} 筆（暖機丟棄 {len(df)-total_cp} 筆）")
print(f"  觸發（FLAG_CP）     ：{cp_trigger} 筆 ({100*cp_trigger/max(total_cp,1):.2f}%)")
print(f"  DIVERGE_COL 峰值   ：{div_vals.max():.1f}% （最大發散程度）")
print(f"  DIVERGE_COL 基線   ：p50={div_vals.median():.1f}%  p90={np.percentile(div_vals.dropna(), 90):.1f}%  p95={np.percentile(div_vals.dropna(), 95):.1f}%")

# Event counting
arr = valid_cp[FLAG_CP].values.astype(int)
ch  = np.diff(np.concatenate([[0], arr, [0]]))
ev_starts = np.where(ch ==  1)[0]
ev_ends   = np.where(ch == -1)[0]
n_events  = len(ev_starts)
ev_dur    = ev_ends - ev_starts

print(f"\n  change point detection events（連續觸發 = 1 次）：{n_events} 次")
if n_events > 0:
    print(f"    平均持續：{ev_dur.mean():.1f} 分鐘")
    print(f"    最短 / 最長：{ev_dur.min()} / {ev_dur.max()} 分鐘")

print(f"\n  🔎 自動診斷：")
p90_div = float(np.percentile(div_vals.dropna(), 90))
if DIVERGE_PCT < p90_div:
    print(f"    ⚠️  閾值 {DIVERGE_PCT}% 低於 p90 發散值（{p90_div:.1f}%）")
    print(f"       正常日夜波動超過你的閾值 → 建議調高至 {p90_div*1.5:.0f}% 以減少日常誤報")
elif n_events == 0:
    print(f"    ⚠️  0 次觸發。閾值 {DIVERGE_PCT}% 高於資料集所有發散值（最大 {div_vals.max():.1f}%）")
    print(f"       建議降低 DIVERGE_PCT 到 {div_vals.max()*0.7:.0f}% 左右開始試探")
elif cp_trigger / total_cp > 0.10:
    print(f"    ⚠️  觸發比例 {100*cp_trigger/total_cp:.1f}% 偏高（>10%），閾值可能設得太低")
else:
    print(f"    ✅ 觸發比例 {100*cp_trigger/total_cp:.2f}%，參數設定合理")

ratio = SHORT_WIN / LONG_WIN
print(f"\n  SHORT/LONG 比例：{ratio:.2f}（建議 ≤ 0.2，你的設定 {'✅ 合理' if ratio <= 0.2 else '⚠️ 偏大，短期均線還不夠平滑'}）")


---
**自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 5：將偵測結果輸出給 Grafana

### 為什麼輸出到 Grafana？

Python 的偵測邏輯是「分析工具」，不是「監控系統」。
要讓偵測結果進入日常維運流程，需要把事件輸出到 Grafana：

```
Python Lab (development / validation)        Production environment
──────────────────────────────────────       ──────────────────────────────────────────
Design and validate detection logic  →  Deployment path 1: Prometheus Alert Rules
                                                             (detection at the PromQL layer)
Detection events output as JSON      →  Deployment path 2: Grafana Annotation API
                                                             (mark anomaly timestamps on Dashboard)
                                     →  Deployment path 3: Alertmanager Webhook
                                                             (push to Slack/PagerDuty)
```

### Grafana Annotation 的用途

Annotation 是 Grafana Dashboard 上的垂直標記線，讓你能：
1. 在任何 Panel 上看到「異常發生時間」和「偵測方法」
2. 與其他指標的時序對比（異常時間和 CPU spike 有沒有相關？）
3. 後續 RCA 時，快速定位事件窗口

**本 Step 的兩個輸出方案：**
- **方案 A（JSON 檔案）**：輸出到 `events.json`，可手動匯入 Grafana 或接 Webhook
- **方案 B（PromQL Alert Rule）**：等效的 PromQL，可直接複製到 Prometheus Alert 設定


In [ ]:
# 方案 A：輸出為 JSON 檔案供 Grafana Annotations 使用
# Grafana 可以從 JSON datasource 或直接 API 匯入 annotation

def build_grafana_annotations(df):
    """將偵測事件轉換為 Grafana annotation 格式。"""
    annotations = []
    
    # Z-score 告警
    for _, row in df[df['flag_zscore_deadband']].iterrows():
        annotations.append({
            "time":     int(row['timestamp'].timestamp() * 1000),
            "text":     f"Z-score alert: Z={row.get('z_score', 0):.2f}，Traffic={row['traffic_total']/1e6:.2f} MB/s",
            "tags":     ["zscore", "anomaly", DEVICE],
            "isRegion": False,
        })
    
    # change point detection events
    for _, row in df[df['change_point']].iterrows():
        annotations.append({
            "time":     int(row['timestamp'].timestamp() * 1000),
            "text":     f"Change-point detection: moving-average divergence {row.get('ma_divergence', 0)*100:.0f}%",
            "tags":     ["changepoint", "anomaly", DEVICE],
            "isRegion": False,
        })
    
    return sorted(annotations, key=lambda x: x['time'])

annotations = build_grafana_annotations(df)
output_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_grafana_annotations.json"
with open(output_path, 'w') as f:
    json.dump(annotations, f, ensure_ascii=False, indent=2)

print(f"Grafana Annotations 輸出：")
print(f"  總事件數：{len(annotations)}")
print(f"  輸出路徑：{output_path}")
# open() accepts Path objects on all platforms
print()
print("前 3 個事件：")
for ann in annotations[:3]:
    ts = pd.Timestamp(ann['time'], unit='ms')
    print(f"  [{ts.strftime('%H:%M')}] {ann['text'][:60]}")

In [ ]:
# 方案 B：PromQL alert rule 等效說明
print("PromQL Alert Rule 等效實作（可直接複製到 Grafana Alert 或 Alertmanager）：")
print()

promql_examples = [
    (
        "fixed threshold alerts",
        'rate(node_network_receive_bytes_total{device!~"lo|docker.*"}[1m]) > 5000000',
        f"Traffic超過 5 MB/s"
    ),
    (
        "Z-score 近似（Grafana Recording Rule）",
        '(rate_1m - avg_over_time(rate_1m[20m])) / stddev_over_time(rate_1m[20m]) > 3',
        "需先定義 recording rule：rate_1m"
    ),
]

for name, rule, note in promql_examples:
    print(f"  {name}：")
    print(f"  {rule}")
    print(f"  （{note}）")
    print()

print("提示：在 Grafana UI → Alerting → Alert rules → New alert rule")
print("      貼上 PromQL 表達式並設定通知 channel")

---
**探索練習** — 修改 deadband 參數，觀察告警數量變化。

---

## 探索練習：調整 Deadband 與 Z-score 閾值

### 實驗 1：調整 Deadband N

```python
N_EXPLORE = 1   # most sensitive; triggers on every Z>3 point
N_EXPLORE = 3   # default; requires 3 consecutive minutes above threshold
N_EXPLORE = 5   # conservative; suitable for environments with high natural traffic variance
N_EXPLORE = 8   # very conservative; triggers only on long-lasting sustained anomalies
```

**觀察：**
- N=1 → 告警數量是多少？有多少是「孤立單點」？
- N=5 → 告警數量減少多少？有沒有遺漏掉重要的異常事件？
- N 的最佳值取決於：你的異常通常至少持續幾分鐘？

### 實驗 2：調整 Z-score 閾值

除了 deadband，也試試調整 `Z_THRESH`（在 Step 2 的 code cell 中）：

```python
Z_THRESH = 2.0  # more sensitive (~5% of points will exceed)
Z_THRESH = 3.0  # standard (~0.3%)
Z_THRESH = 4.0  # conservative (~0.006%)
```

### 觀察矩陣

執行不同組合後，記錄告警數量：

```
           N=1   N=3   N=5
Z > 2.0  [  ?  ] [  ?  ] [  ?  ]
Z > 3.0  [  ?  ] [  ?  ] [  ?  ]
Z > 4.0  [  ?  ] [  ?  ] [  ?  ]
```

哪個組合的告警數最接近「每天 5 次」（合理的告警頻率）？


In [ ]:
# 🔧 修改這裡！
N_EXPLORE = 3  # 試試 1, 3, 5, 8

flag_explore = (
    df['flag_zscore_raw'].astype(int)
    .rolling(window=N_EXPLORE, min_periods=N_EXPLORE)
    .sum() >= N_EXPLORE
)

results = []
for n in [1, 2, 3, 5, 8]:
    f = (df['flag_zscore_raw'].astype(int)
         .rolling(window=n, min_periods=n)
         .sum() >= n)
    results.append({'deadband_N': n, '告警筆數': f.sum()})

df_result = pd.DataFrame(results)
print("Deadband 效果對照表：")
print(df_result.to_string(index=False))

print(f"\n你選擇的 N={N_EXPLORE}：{flag_explore.sum()} 筆告警")
print("\n注意：告警多 ≠ 壞，告警少 ≠ 好")
print("      關鍵在於：重要事件是否有被捕捉到？")

---
**討論暫停 ▸ 全班討論** — 暫停執行，與全班分享你的觀察。

---

## ⚠ 人類驗證點 #2 — 閾值是業務決定，不是技術決定

### 判斷標準

| 問題 | 你需要思考的面向 |
|------|----------------|
| Z > 3 是你的異常嗎？ | 取決於你的業務 SLA 和流量的重尾程度 |
| Deadband N=3 適合嗎？ | 取決於異常通常持續多久（比 scrape_interval 長多少？）|
| 誤警報率 vs 漏警報率 | 你的組織更怕哪個？醫療 vs 一般 IT 的答案截然不同 |
| 固定閾值值該設多高？ | 來自對業務流量容量的理解，不是統計 |

---

### 閾值設定決策框架

```
Step 1: Find the absolute upper bound
  Query your link capacity (e.g. 1 Gbps = 125 MB/s)
  Fixed threshold at 80% (100 MB/s) is a safe starting point

Step 2: Understand the normal peak
  Query the 95th percentile over the past 30 days
  Set the Z-score threshold at "2–3σ above the normal peak"

Step 3: Assess the cost of anomalies
  Is missing an alert more costly than sending a false alarm?
    → Lower the threshold (more sensitive)
  Is alert fatigue already degrading team attention?
    → Increase deadband N (reduce alert count)

Step 4: Validate against data
  Replay known past events (real anomalies); confirm your threshold triggers
  Count alerts over the past 7 days; tune until fewer than 5–10 per day
```

---

### 討論問題

> 「你的業務場景中，漏掉一個告警（False Negative）和多發一個假警報（False Positive），哪個代價更大？」

> 「如果流量固定 95th percentile 是 5 MB/s，你的固定閾值應該設多高？為什麼不直接用 5 MB/s？」

> 「change point detection到了什麼？那是真的異常嗎？你會怎麼確認？」

> 「如果你有一個鏈路在每天凌晨 2 點固定執行備份（流量突然高 3 倍），Z-score 會誤觸發嗎？你怎麼處理？」

---

### 關鍵結論

```
Technology can detect deviations,
but "how large a deviation counts as anomalous" is a business question.

Z > 3 is statistically rare; it does not automatically require business action.
Threshold design requires human business-value judgment,
followed by data validation to confirm that judgment is sound.
```


In [ ]:
# 整合偵測事件輸出 — 供 Lab 08 Capstone 使用
events = []
for _, row in df[df['flag_zscore_deadband']].iterrows():
    events.append({
        'timestamp': row['timestamp'],
        'device': DEVICE,
        'type': 'zscore',
        'severity': 'high' if abs(row.get('z_score', 0)) > 4 else 'medium',
        'z_score': round(row.get('z_score', 0), 2),
        'traffic_total_mb': round(row['traffic_total'] / 1e6, 3),
    })
for _, row in df[df['change_point']].iterrows():
    events.append({
        'timestamp': row['timestamp'],
        'device': DEVICE,
        'type': 'changepoint',
        'severity': 'medium',
        'z_score': 0,
        'traffic_total_mb': round(row['traffic_total'] / 1e6, 3),
    })

df_events = pd.DataFrame(events).sort_values('timestamp').reset_index(drop=True)
_ev_path = PROJECT_ROOT / "outputs" / "workshop" / "workshop_lab02_events.csv"
df_events.to_csv(_ev_path, index=False)

print(f"偵測事件摘要：")
print(f"  Z-score 事件：{(df_events['type'] == 'zscore').sum()} 筆")
print(f"  change point detection events：    {(df_events['type'] == 'changepoint').sum()} 筆")
print(f"  High severity：{(df_events['severity'] == 'high').sum()} 筆")
print(f"\n✅ 事件資料已儲存到 {_ev_path}")
print("Lab 02 完成！接下來進入 Lab 08 — AI-SPC + Agentic AI Capstone。")